# System loop testing

In [1]:
import base64
import glob
from time import time
import fal_client
import os
import requests
from google.genai import types

import matplotlib.pyplot as plt
import numpy as np

from dotenv import load_dotenv
from openai import AzureOpenAI

from google import genai
from dotenv import load_dotenv
from io import BytesIO
from IPython.display import Video
from PIL import Image

load_dotenv("../../../../_starter/.env")

True

# Google client

In [2]:
google_client = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))

# Azure client

In [3]:
client = AzureOpenAI(
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_version=os.getenv("AZURE_OPENAI_API_VERSION"),
)

deployment_name = "gpt-4.1"

# Convert image to base64

In [4]:
# Convert image to base64 for API
def image_to_base64(image):
    buffered = BytesIO()
    image.save(buffered, format="JPEG")
    img_str = base64.b64encode(buffered.getvalue()).decode()
    return img_str

# Subject

In [5]:
subject = "Pokemon"
subject = subject.lower()

# Assistant 1 - creator

In [6]:
with open("../1_prompt_thinking.md", "r", encoding="utf-8") as f:
    prompt_creator = f.read()

prompt_creator = prompt_creator.format(subject=subject)


def get_reply_with_image(conv_hist, image):
    img_base64 = image_to_base64(image)

    conv_hist[1] = {
        "role": "user",
        "content": [
            {
                "type": "image_url",
                "image_url": {
                    "url": f"data:image/jpeg;base64,{img_base64}"
                }
            }
        ]
    }

    response = client.chat.completions.create(
        model=deployment_name,
        messages=conv_hist
    )

    response_text = response.choices[0].message.content
    print(response_text)
    return response_text

def add_self_to_conv_hist(message):
    return {
        "role": "assistant",
        "content": [
            {
                "type": "text",
                "text": message
            }
        ]
    }

def add_reply_to_conv_hist(message):
    return {
        "role": "user",
        "content": [
            {
                "type": "text",
                "text": message
            }
        ]
    }

# Assistant 2 - discriminator

In [7]:
def discriminate(text_to_verify, subject):
    with open("../2_project_discriminator.md", "r", encoding="utf-8") as f:
        prompt = f.read()

    prompt = prompt.format(subject=subject)

    messages = [
        {
            "role": "system",
            "content": [
                {
                    "type": "text",
                    "text": prompt
                }
            ]
        },
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": text_to_verify
                }
            ]
        }
    ]

    response = client.chat.completions.create(
        model=deployment_name,
        messages=messages
    )

    response_text = response.choices[0].message.content
    discrimination = str(response_text).lower()
    if "false" in discrimination:
        print(f"{subject} was not detected - passing")
        return False
    else:
        print(f"{subject} was detected - failing")
        return True

# Assistant 3 - artist

In [8]:
def draw(message, image, folder, turn):
    response = google_client.models.generate_content(
        model="gemini-2.5-flash-image",
        contents=[message, image],
        config=types.GenerateContentConfig(
            temperature=0,
            response_modalities=[
                "IMAGE",
            ],
            image_config=types.ImageConfig(
                aspect_ratio="1:1",
            ),
        ),
    )

    for part in response.candidates[0].content.parts:
        if part.text is not None:
            print(part.text)
        elif part.inline_data is not None:
            image = Image.open(BytesIO(part.inline_data.data))
            if not os.path.exists(f"outputs/{folder}"):
                os.makedirs(f"outputs/{folder}")
            image.save(f"outputs/{folder}/test_{turn}.png")

    return image

# Assistant 4 - guesser

In [9]:
def guess(guessed_subjects, image):
    with open("../4_guesser.md", "r", encoding="utf-8") as f:
        prompt = f.read()

    img_base64 = image_to_base64(image)

    messages = [
    {
        "role": "system",
        "content": [
            {
                "type": "text",
                "text": prompt
            }
        ]
    },
    {
        "role": "user",
        "content": [
            {
                "type": "text",
                "text": f"Attempted guesses: {guessed_subjects}"
            },
            {
                "type": "image_url",
                "image_url": {
                    "url": f"data:image/jpeg;base64,{img_base64}"
                }
            }
        ]
    }
]

    response = client.chat.completions.create(
        model=deployment_name,
        messages=messages
    )

    response_text = response.choices[0].message.content
    print(response_text)
    return response_text.lower()

# Loop

In [10]:
image_paths = glob.glob("assets/*")
image_paths

['assets/5.jpg', 'assets/2.jpg', 'assets/3.jpg', 'assets/1.jpg']

In [11]:
image_paths

['assets/5.jpg', 'assets/2.jpg', 'assets/3.jpg', 'assets/1.jpg']

In [12]:
for i in range(2):
    for image_path in image_paths:
        c = 0
        image_state = Image.open(image_path)
        timestamp = int(time())

        img_base64 = image_to_base64(image_state)

        agent_1_conv_hist = [
            {
                "role": "system",
                "content": [
                    {
                        "type": "text",
                        "text": prompt_creator
                    }
                ]
            },
            {
                "role": "user",
                "content": [
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": f"data:image/jpeg;base64,{img_base64}"
                        }
                    }
                ]
            }
        ]

        if not os.path.exists(f"outputs/{timestamp}"):
            os.makedirs(f"outputs/{timestamp}")

        image_state.save(f"outputs/{timestamp}/test_0.png")
        guessed_subjects = ""

        while c < 10:
            c += 1
            
            print(f"Agent 1 - Creator - Turn {c}")
            agent_1_response = get_reply_with_image(agent_1_conv_hist, image_state)
            agent_1_conv_hist.append(add_self_to_conv_hist(agent_1_response))
            print("--------------------------------")

            print(f"Agent 2 - Discriminator - Turn {c}")
            if not(discriminate(agent_1_response, subject)):
                print(f"Prompt is not describing {subject} - passing!")
                print("--------------------------------")

                print(f"Agent 3 - Artist - Turn {c}")
                image_state = draw(agent_1_response, image_state, timestamp, c)
                print("--------------------------------")

                print(f"Agent 4 - Guesser - Turn {c}")
                guessed = guess(guessed_subjects, image_state)
                guessed_subjects += f"{guessed}, "
                
                if guessed == subject:
                    print(f"Guessed {subject} in {c} turns")
                    print("--------------------------------")
                    break
                else:
                    print(f"Guessed {guessed} - try again!")
                    agent_1_conv_hist.append(add_reply_to_conv_hist(f"User guess that this is {guessed} - try again!"))
                    print("--------------------------------")
            else:
                print(f"Prompt is describing {subject} - try again!")
                agent_1_conv_hist.append(add_reply_to_conv_hist("Prompt is describing {subject} - try again!"))
                c -= 1
                pass

Agent 1 - Creator - Turn 1
Add two small white circles inside the upper half of the purple semicircle, spaced apart horizontally, to create a sense of symmetry.
--------------------------------
Agent 2 - Discriminator - Turn 1
pokemon was not detected - passing
Prompt is not describing pokemon - passing!
--------------------------------
Agent 3 - Artist - Turn 1
--------------------------------
Agent 4 - Guesser - Turn 1
Ghost
Guessed ghost - try again!
--------------------------------
Agent 1 - Creator - Turn 2
Extend three small, curved shapes downward from the flat bottom edge of the purple semicircle to resemble wispy, uneven extensions. Add a soft, hazy outline around the entire purple shape to give it a slightly blurred, misty appearance.
--------------------------------
Agent 2 - Discriminator - Turn 2
pokemon was detected - failing
Prompt is describing pokemon - try again!
Agent 1 - Creator - Turn 2
To add more detail, place a small, jagged, light lavender oval slightly below t

KeyboardInterrupt: 

# Create gifs for all folders

In [13]:
folders = glob.glob("outputs/*/")

for folder in folders:
    folder = folder[:-1]
    image_paths = glob.glob(f"{folder}/*.png")
    image_paths = sorted(image_paths, key=lambda x: int(x.split("_")[-1].split(".")[0]))
    images = [Image.open(image) for image in image_paths]
    images[0].save(f"{folder}.gif", save_all=True, append_images=images[1:], duration=100, loop=0)
    print(f"Created gif for {folder}")

Created gif for outputs/1760641896
Created gif for outputs/1760644321
Created gif for outputs/1760645697
Created gif for outputs/1760645253
Created gif for outputs/1760645546
Created gif for outputs/1760644481
Created gif for outputs/1760645395
Created gif for outputs/1760643651
Created gif for outputs/1760645847
Created gif for outputs/1760643993
Created gif for outputs/1760641572
Created gif for outputs/1760642332
Created gif for outputs/1760643016
Created gif for outputs/1760646149
Created gif for outputs/1760645103
Created gif for outputs/1760642975
Created gif for outputs/1760645991


# Render last images in style of Pokemon with background matching the theme of the image

In [22]:
def render_pokemon(folder, image_path):
    prompt = f"""
    Render this sketch of a Pokemon in style of Pokemon anime illustration with background matching the theme of the image
    """

    image = Image.open(image_path)

    response = google_client.models.generate_content(
        model="gemini-2.5-flash-image",
        contents=[prompt, image],
        config=types.GenerateContentConfig(
            temperature=0,
            response_modalities=[
                "IMAGE",
            ],
            image_config=types.ImageConfig(
                aspect_ratio="1:1",
            ),
        ),
    )

    for part in response.candidates[0].content.parts:
        if part.text is not None:
            print(part.text)
        elif part.inline_data is not None:
            image = Image.open(BytesIO(part.inline_data.data))
            folder = folder.replace("outputs", "renders")
            image.save(f"{folder}.png")

    return image

In [24]:
folders = glob.glob("outputs/*/")

for folder in folders[1:]:
    folder = folder[:-1]
    image_paths = glob.glob(f"{folder}/*.png")
    image_paths = sorted(image_paths, key=lambda x: int(x.split("_")[-1].split(".")[0]))
    last_image = image_paths[-1]
    render_pokemon(folder, last_image)